In [8]:
!pip install --force-reinstall langchain torch transformers nltk wikipedia sentence-transformers statistics

  Using cached wikipedia-1.4.0-py3-none-any.whl
  Using cached statistics-1.0.3.5-py3-none-any.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [2]:
import nltk
import transformers
import numpy as np
import nltk
import re
import wikipedia
import statistics
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM

from warnings import filterwarnings; filterwarnings("ignore")

In [3]:
#Task 1
task1_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for beginners"
)

print(task1_template.format(topic="Artificial Intelligence"))

Explain Artificial Intelligence in simple terms for beginners


In [4]:
#Task 2
task2_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} tone"
)

# Test cases
print(task2_template.format(topic="AI", audience="beginners", tone="friendly"))
print(task2_template.format(topic="Python", audience="kids", tone="fun"))
print(task2_template.format(topic="Deep Learning", audience="engineers", tone="technical"))


Explain AI for beginners in a friendly tone
Explain Python for kids in a fun tone
Explain Deep Learning for engineers in a technical tone


In [5]:
teaching_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} clearly step by step"
)

interview_template = PromptTemplate(
    input_variables=["topic"],
    template="Ask 3 questions about {topic}"
)

story_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} as a story"
)

print(teaching_template.format(topic="Machine Learning"))
print(interview_template.format(topic="Machine Learning"))
print(story_template.format(topic="Machine Learning"))

Explain Machine Learning clearly step by step
Ask 3 questions about Machine Learning
Explain Machine Learning as a story


In [6]:
def get_chat_prompt(role, topic):
    if role == "teacher":
        system_template = "You are a helpful teacher. Explain concepts clearly."
    elif role == "interviewer":
        system_template = "You are an interviewer. Ask insightful questions."
    elif role == "motivator":
        system_template = "You are a motivator. Inspire the learner."
    else:
        system_template = "You are a helpful assistant."

    chat_template = ChatPromptTemplate.from_messages([
        ("system", system_template),
        ("user", "{topic}")
    ])

    return chat_template.format_messages(topic=topic)

print(get_chat_prompt("teacher", "Neural Networks"))

[SystemMessage(content='You are a helpful teacher. Explain concepts clearly.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Neural Networks', additional_kwargs={}, response_metadata={})]


In [7]:
def validate_inputs(audience, tone):
    valid_audience = ["beginner", "intermediate", "expert"]
    valid_tone = ["formal", "casual", "fun"]

    if audience not in valid_audience:
        audience = "beginner"

    if tone not in valid_tone:
        tone = "formal"

    return audience, tone

# Test validation
print(validate_inputs("kids", "fun"))


('beginner', 'fun')


In [8]:
styles = {
    "teaching": "Explain {topic} for {audience} in a {tone} teaching style",
    "interview": "Ask interview questions about {topic} for a {audience} in a {tone} tone",
    "storytelling": "Explain {topic} for {audience} in a {tone} storytelling style"
}


def generate_prompt(topic, audience, tone, style):
    audience, tone = validate_inputs(audience, tone)

    template_string = styles.get(style)

    prompt_template = PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template=template_string
    )

    return prompt_template.format(topic=topic, audience=audience, tone=tone)

# Example
print(generate_prompt("Neural Networks", "beginner", "fun", "storytelling"))


Explain Neural Networks for beginner in a fun storytelling style


In [9]:
reusable_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} tone"
)

inputs = [
    ("AI", "beginner", "fun"),
    ("Blockchain", "expert", "formal"),
    ("Python", "intermediate", "casual"),
    ("Data Science", "beginner", "formal"),
    ("Neural Networks", "expert", "technical")
]

for topic, audience, tone in inputs:
    print(reusable_template.format(topic=topic, audience=audience, tone=tone))

Explain AI for beginner in a fun tone
Explain Blockchain for expert in a formal tone
Explain Python for intermediate in a casual tone
Explain Data Science for beginner in a formal tone
Explain Neural Networks for expert in a technical tone


## Simple Robustness Test and Impossibility Tests

In [10]:
# Test extreme and edge-case inputs
stress_inputs = [
    ("", "beginner", "fun"),
    ("A"*1000, "expert", "formal"),
    ("123456", "intermediate", "casual"),
    ("!@#$%^&*()", "beginner", "fun")
]

print("Results of Stress Testing: \n")
for topic, audience, tone in stress_inputs:
    try:
        result = reusable_template.format(topic=topic, audience=audience, tone=tone)
        print(result[:100])
    except Exception as e:
        print("Error:", e)


Results of Stress Testing: 

Explain  for beginner in a fun tone
Explain AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA
Explain 123456 for intermediate in a casual tone
Explain !@#$%^&*() for beginner in a fun tone


In [11]:
invalid_inputs = [
    (None, "beginner", "fun"),
    ("AI", None, "formal"),
    ("ML", "unknown", "weird"),
]

print("Results of Impossibility Testing: \n")
for topic, audience, tone in invalid_inputs:
    try:
        audience, tone = validate_inputs(audience, tone)
        result = reusable_template.format(topic=str(topic), audience=audience, tone=tone)
        print(result)
    except Exception as e:
        print("Handled Error:", e)


Results of Impossibility Testing: 

Explain None for beginner in a fun tone
Explain AI for beginner in a formal tone
Explain ML for beginner in a formal tone


In [12]:
# Measure variability and consistency of outputs
outputs = []

test_topics = ["AI", "ML", "DL", "NLP", "CV"]

for t in test_topics:
    outputs.append(reusable_template.format(topic=t, audience="beginner", tone="formal"))

# Basic statistics
lengths = [len(o) for o in outputs]

print("Results of Statistical Analysis: \n")
print("Mean Length:", statistics.mean(lengths))
print("Std Dev:", statistics.stdev(lengths))
print("Min Length:", min(lengths))
print("Max Length:", max(lengths))

# Interpretation:
# Low std deviation -> consistent structure
# High variance -> template instability

Results of Statistical Analysis: 

Mean Length: 40.2
Std Dev: 0.4472135954999579
Min Length: 40
Max Length: 41
